In [ ]:
# EXP (no es del baseline): Kaggle monta los inputs en otra carpeta cuando el notebook se crea por API.
# Arma /tmp/lab/input/<nombre> -> carpeta real y se para en /tmp/lab/work, asi '../input/<nombre>/...' del baseline funciona sin tocarlo.
import os, glob
def _buscar(nombre, archivo):
    for prof in range(0, 5):
        patron = "/kaggle/input/" + "*/" * prof
        for d in glob.glob(patron):
            d = d.rstrip("/")
            if os.path.basename(d).endswith(nombre) and os.path.exists(os.path.join(d, archivo)):
                return d
    raise FileNotFoundError(f"{nombre}/{archivo} no aparece en /kaggle/input: {sorted(glob.glob('/kaggle/input/*/*'))[:50]}")
_inputs = {"petfinder-adoption-prediction": "train/train.csv",
           "examen-lab-ii-preprocesado-texto": "train.parquet",
           "examen-lab-ii-preprocesado-img": "train.parquet"}
if not all(os.path.exists(f"../input/{n}/{a}") for n, a in _inputs.items()):
    os.makedirs("/tmp/lab/input", exist_ok=True)
    os.makedirs("/tmp/lab/work", exist_ok=True)
    for n, a in _inputs.items():
        destino = f"/tmp/lab/input/{n}"
        if not os.path.exists(destino):
            os.symlink(_buscar(n, a), destino)
    os.chdir("/tmp/lab/work")
print({n: os.path.realpath(f"../input/{n}") for n in _inputs})

In [ ]:
# Importación de Librerías
import pandas as pd 

from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from lightgbm import LGBMClassifier, early_stopping

def metric(y_true, y_pred):
    res = cohen_kappa_score(y_true, y_pred.reshape((y_true.shape[0], 5), order="F").argmax(axis=1), weights= 'quadratic')
    return "kappa", res, True

In [ ]:
train = pd.read_csv('../input/petfinder-adoption-prediction/train/train.csv').set_index("PetID")
test = pd.read_csv('../input/petfinder-adoption-prediction/test/test.csv').set_index("PetID")
from sklearn.model_selection import StratifiedGroupKFold
folds = [train.index[test_idx] for train_idx, test_idx in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(train, train["AdoptionSpeed"], groups=train["RescuerID"])]

In [ ]:
for c in ['Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2',
       'Color3', 'FurLength', 'MaturitySize', 'Vaccinated', 'Dewormed',
       'Sterilized', 'Health', 'State', 'RescuerID']:
    train[c] = train[c].astype("category")    
    test[c] = test[c].astype("category")

In [ ]:
train = train\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/train.parquet"))\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-img/train.parquet"))\
    .select_dtypes(exclude="O")
train

In [ ]:
test = test\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/test.parquet"))\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-img/test.parquet"))\
    .select_dtypes(exclude="O")
test

In [ ]:
fold_preds = []
test_preds = []
for i, fold in enumerate(folds):
    X_train = train.drop(fold).drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_train = train.drop(fold)["AdoptionSpeed"]

    X_valid = train.loc[fold].drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_valid = train.loc[fold, "AdoptionSpeed"]

    learner = LGBMClassifier()
    learner.fit(X_train, y_train, )
    
    learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                eval_metric= metric, callbacks=[early_stopping(5)])
    fold_preds.append(pd.Series(learner.predict(X_valid), index=fold, name="AdoptionSpeed"))
    test_preds.append(pd.DataFrame(learner.predict_proba(test[X_train.columns]), index=test.index))

fold_preds = pd.concat(fold_preds)
test_preds = sum(test_preds).idxmax(axis=1).rename("AdoptionSpeed")

In [ ]:
res = pd.DataFrame({"baseline_grupos": [cohen_kappa_score(train.loc[f, "AdoptionSpeed"], fold_preds.loc[f], weights="quadratic") for f in folds]},
                   index=[f"fold_{i}" for i in range(len(folds))])
print(res.round(4).to_string())
print("media", round(res["baseline_grupos"].mean(), 4), "desvio", round(res["baseline_grupos"].std(), 4))

In [ ]:
test_preds.to_csv("submission.csv")
!head submission.csv

In [ ]:
# EXP (no es del baseline): si se cambio de carpeta, dejar submission.csv en el output de Kaggle
import shutil
if os.getcwd() != "/kaggle/working" and os.path.exists("submission.csv"):
    shutil.copy("submission.csv", "/kaggle/working/submission.csv")